# **Investment Portfolio Management**

## **About the scenario**
This scenario demonstrates best practices in a data engineering context, where notebooks are used to orchestrate complex tasks involving real-time data fetching, computation, and report delivery. The notebook is modular, and secure, employing batch processing, error handling, and proper separation of concerns.

In this scenario, we will:

1. Ingest a CSV file containing the user’s investment portfolio.
2. Fetch real-time stock prices via an API using *Function Calling*.
3. Perform calculations on the portfolio, including total value and percentage changes, using *Code Interpreter*.
4. Generate a detailed report summarizing the portfolio's performance.
5. Save the report to Azure Blob Storage using *Function Calling*.

## **Azure OpenAI Setup**

#### 1 - Retrieve and set secrets


In [ ]:
import os
import time
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv(dotenv_path=".env/.development.env")

# Retrieve the secrets
__AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
__AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
__AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
__AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")

# DEBUG: Print to verify
print("API Key:", __AZURE_OPENAI_API_KEY)
print("Endpoint:", __AZURE_OPENAI_ENDPOINT)
print("Deployment:", __AZURE_OPENAI_DEPLOYMENT)
print("API Key:", __AZURE_OPENAI_API_VERSION)

#### 2 - Create Azure OpenAI client

In [ ]:
from openai import AzureOpenAI

# Create an instance of the AzureOpenAI client
openai_client = AzureOpenAI(
    api_key=__AZURE_OPENAI_API_KEY,  
    api_version=__AZURE_OPENAI_API_VERSION,
    azure_endpoint=__AZURE_OPENAI_ENDPOINT
)

### 3 - Upload Support File(s)
Sample personal investment portfolio file.

In [ ]:
# Directory containing files to upload
directory = "data"
purpose = 'assistants'

# Ensure directory exists
if not os.path.isdir(directory):
    print(f"Directory '{directory}' does not exist.")
else:
    # Iterate over each file in the directory
    for file_name in os.listdir(directory):
        file_path = os.path.join(directory, file_name)
        
        # Skip if not a file
        if not os.path.isfile(file_path):
            continue

        # Check if the file already exists in OpenAI
        existing_files = openai_client.files.list()  # List all files
        file_exists = False
        for f in existing_files:
            if f.filename == file_name and f.purpose == purpose:
                # Delete the existing file
                openai_client.files.delete(file_id=f.id)
                print(f"Deleted existing file: {file_name}")
                file_exists = True
                break

        # Upload the new file
        with open(file_path, "rb") as file_data:
            portfolio_file = openai_client.files.create(
                file=file_data,
                purpose=purpose
            )
        print(f"Uploaded file: {file_name}")


In [ ]:
# Upload a file with an "assistants" purpose
portfolio_file = openai_client.files.create(
  file=open("data/investment_portfolio.csv", "rb"),
  purpose='assistants'
)

## **Assistant**

### 1 - Configure OpenAI Assistant

In [ ]:
# Create an assistant using the file ID with code interpreter tool enabled
assistant = openai_client.beta.assistants.create(
  name="Investment Management Assistant",
  instructions=
    f"You are an expert investment analyst. Use your knowledge base to answer questions about personal investment portfolio management.",
  model=__AZURE_OPENAI_DEPLOYMENT,
  tools=[
    {"type": "code_interpreter"},
  ],
  tool_resources={
    "code_interpreter":{"file_ids":[portfolio_file.id]}
  }
)

### 2 - Create Thread

In [ ]:
# Create a thread
thread = openai_client.beta.threads.create()
print(thread)

### 3 - Create Message

In [ ]:
prompt_content = "What ticker symbol has the most stock? Which has the most investment?"

# Add a user question to the thread
message = openai_client.beta.threads.messages.create(
    thread_id=thread.id,
    role="user",
    content=prompt_content
)

print(message)

### 4 - Run Thread

In [ ]:
run = openai_client.beta.threads.runs.create(
  thread_id=thread.id,
  assistant_id=assistant.id,
  instructions=prompt_content,
)

print(run)

### 5 - Retrieve Status

In [ ]:
import time

while True:
    time.sleep(62)

    # Retrieve the run status
    run_status = openai_client.beta.threads.runs.retrieve(
        thread_id=thread.id,
        run_id=run.id
    )
    print(run_status.model_dump_json(indent=4))

    if run.status == "completed":
        messages = openai_client.beta.threads.runs.list_messages(thread_id=thread.id)

        # Loop through messages and print content based on role
        for msg in messages.data:
            role = msg.role
            content = msg.content[0].text.value
            print(f"{role.capitalize()}: {content}")
        break
    else:
            print("Waiting for the Assistant to process...")
            time.sleep(62)

## Clean Up

In [ ]:
response = openai_client.beta.assistants.delete(assistant.id)
print(response)

----------------

## Scratch Pad

In [ ]:
# File details
#file_name = "data/investment_portfolio.csv"
#purpose = 'assistants'

# Check if the file already exists in OpenAI
#existing_files = openai_client.files.list()  # List all files
#file_exists = any(f.filename == os.path.basename(file_name) and f.purpose == purpose for f in existing_files)

#if not file_exists:
    # Upload the file if it doesn't exist
    #with open(file_name, "rb") as file_data:
        #portfolio_file = openai_client.files.create(
            #file=file_data,
            #purpose=purpose
        #)
    #print("File uploaded successfully.")
#else:
    #print("File already exists; skipping upload.")

In [ ]:
import time
from IPython.display import clear_output

start_time = time.time()

status = run.status

while status not in ["completed", "cancelled", "expired", "failed"]:
    time.sleep(5)
    run = openai_client.beta.threads.runs.retrieve(thread_id=thread.id,run_id=run.id)
    print("Elapsed time: {} minutes {} seconds".format(int((time.time() - start_time) // 60), int((time.time() - start_time) % 60)))
    status = run.status
    print(f'Status: {status}')
    clear_output(wait=True)

messages = openai_client.beta.threads.messages.list(
  thread_id=thread.id
) 

print(f'Status: {status}')
print("Elapsed time: {} minutes {} seconds".format(int((time.time() - start_time) // 60), int((time.time() - start_time) % 60)))
print(messages.model_dump_json(indent=2))

Everything after deployment --- everything else should be in notebook (data plane - private endpoint) --- before should not, its control plane (public only)

1. Portal Creation for Deployment (note iteration two, bicep code??)
2. Assistant Creation (API)
    - code to do it not portal (delete for cleanup)
3. Create Thread, Message, Thread run (instructs assistant to create messages) - wait for complete (look at SDK to do this), Listing Messages, Check history of the thread (will see my message, and one from assistant) first message response will be answer (POST/THREADS, POST/THREADS Messages, POST Threads thr)
    - POST /threads
    - POST /threads/:thread_id/messages
    - POST /threads/:thread_id/runs
    - GET /threads/:thread_id/runs/:run_id
    - GET /threads/:thread_id/messages
4. 